# Verificación del tamaño de datos — Parte A, punto 2

Verifica el rango configurado, el número de registros de los CSV mensuales y las estaciones observadas, sin exportar datos ni mostrar viajes individuales. Complementa la auditoría anterior con una ejecución independiente que usa únicamente la biblioteca estándar. Los conteos corresponden a datos obtenidos, antes de limpieza o deduplicación; no certifican calidad ni capacidad histórica.

In [1]:
import ast
import csv
from pathlib import Path

root = Path.cwd().resolve()
while not (root / 'proyecto_mibici' / 'config.py').is_file() and root != root.parent:
    root = root.parent
config_path = root / 'proyecto_mibici' / 'config.py'
config_tree = ast.parse(config_path.read_text(encoding='utf-8'))
parameter_names = {
    'DEFAULT_ANIO_INICIO', 'DEFAULT_MES_INICIO',
    'DEFAULT_ANIO_FIN', 'DEFAULT_MES_FIN',
}
parameters = {}
for statement in config_tree.body:
    if isinstance(statement, ast.Assign):
        for target in statement.targets:
            if isinstance(target, ast.Name) and target.id in parameter_names:
                parameters[target.id] = ast.literal_eval(statement.value)
if parameters.keys() != parameter_names:
    raise ValueError('No se encontraron todos los límites del rango en config.py')
start = parameters['DEFAULT_ANIO_INICIO'] * 12 + parameters['DEFAULT_MES_INICIO'] - 1
end = parameters['DEFAULT_ANIO_FIN'] * 12 + parameters['DEFAULT_MES_FIN'] - 1
if start > end:
    raise ValueError('Rango de fechas invertido')
raw_dir = root / 'data' / 'raw'
months = [divmod(value, 12) for value in range(start, end + 1)]
trip_files = [raw_dir / f'datos_abiertos_{year}_{month + 1:02d}.csv' for year, month in months]
missing = [path.name for path in trip_files if not path.is_file()]
if missing:
    raise FileNotFoundError(missing)
station_file = raw_dir / 'nomenclatura_2026_09.csv'


## Método
Se cuentan registros CSV no vacíos, no líneas físicas. Las estaciones se cuentan como IDs distintos no vacíos en la unión de origen y destino. Solo se retienen IDs de estaciones y conteos agregados. La sustitución de caracteres de decodificación reproduce la auditoría anterior y no altera los campos de ID ASCII seleccionados; no es una normalización de las columnas demográficas ni una auditoría de codificación. Las constantes de fechas se leen de config.py mediante AST para evitar duplicarlas y no importar dependencias de ejecución ajenas a esta verificación.

In [2]:
origins = set()
destinations = set()
monthly_counts = []
for path in trip_files:
    with path.open(encoding='utf-8-sig', errors='replace', newline='') as handle:
        reader = csv.reader(handle)
        header = next(reader)
        origin_index = header.index('Origen_Id')
        destination_index = header.index('Destino_Id')
        count = 0
        for row in reader:
            if not row:
                continue
            if len(row) != len(header):
                raise ValueError(f'Estructura CSV inconsistente en {path.name}, registro {count + 1}')
            count += 1
            origin = row[origin_index].strip()
            destination = row[destination_index].strip()
            if origin:
                origins.add(origin)
            if destination:
                destinations.add(destination)
    monthly_counts.append((path.stem.removeprefix('datos_abiertos_'), count))
observed = origins | destinations
with station_file.open(encoding='utf-8-sig', errors='replace', newline='') as handle:
    station_reader = csv.DictReader(handle)
    station_columns = station_reader.fieldnames
    station_rows = list(station_reader)
catalog_ids = {row['id'].strip() for row in station_rows if row['id'].strip()}
print(f'Meses: {len(monthly_counts)}')
print(f'Viajes obtenidos: {sum(count for _, count in monthly_counts):,}')
print(f'Estaciones de origen: {len(origins)}')
print(f'Estaciones de destino: {len(destinations)}')
print(f'Estaciones observadas (unión): {len(observed)}')
print(f'Filas de nomenclatura: {len(station_rows)}')
print(f'IDs distintos de nomenclatura: {len(catalog_ids)}')
print(f'Estaciones observadas sin correspondencia: {len(observed - catalog_ids)}')
print(f'Estaciones de nomenclatura sin viajes en el rango: {len(catalog_ids - observed)}')
print('Columnas de nomenclatura:', ', '.join(station_columns))
print('Mes | viajes obtenidos')
for month, count in monthly_counts:
    print(f'{month} | {count:,}')


Meses: 36
Viajes obtenidos: 13,621,885
Estaciones de origen: 473
Estaciones de destino: 473
Estaciones observadas (unión): 473
Filas de nomenclatura: 484
IDs distintos de nomenclatura: 484
Estaciones observadas sin correspondencia: 2
Estaciones de nomenclatura sin viajes en el rango: 13
Columnas de nomenclatura: id, name, latitude, longitude, dpcapacity
Mes | viajes obtenidos
2023_10 | 383,239
2023_11 | 384,464
2023_12 | 314,164
2024_01 | 391,591
2024_02 | 383,732
2024_03 | 390,513
2024_04 | 405,308
2024_05 | 410,806
2024_06 | 365,000
2024_07 | 364,177
2024_08 | 394,894
2024_09 | 390,072
2024_10 | 418,690
2024_11 | 391,089
2024_12 | 340,049
2025_01 | 380,184
2025_02 | 372,529
2025_03 | 405,110
2025_04 | 366,193
2025_05 | 395,891
2025_06 | 356,611
2025_07 | 368,186
2025_08 | 374,727
2025_09 | 376,789
2025_10 | 420,064
2025_11 | 380,328
2025_12 | 335,420
2026_01 | 357,180
2026_02 | 333,756
2026_03 | 376,243
2026_04 | 351,851
2026_05 | 315,895
2026_06 | 355,303
2026_07 | 388,257
2026_08 |

## Interpretación
El total de viajes cuenta registros de los archivos del rango, sin deduplicación ni filtros por fechas internas. El total de estaciones observadas no equivale a estaciones activas todos los meses. La nomenclatura es una fotografía de septiembre de 2026 y no certifica las capacidades de 2023–2025. Las estaciones sin correspondencia deben conciliarse antes de asignarles capacidad o coordenadas; no se les imputa capacidad en esta auditoría.